# Gold Layer - Alternative Fuel Infrastructure

## Objective
This notebook creates an analytical gold table from the silver alternative fuel stations dataset.

## Main goals
- aggregate station data into a business-friendly analytical table
- support analysis of alternative fuel infrastructure by location and fuel type
- provide a simplified view of infrastructure readiness across states

## Notes
The gold layer focuses on analytical value, so this table contains aggregated station metrics instead of detailed station-level records.

In [0]:
from pyspark.sql import functions as F

## Imports and configuration

Define the source and target tables used in this notebook.

In [0]:
catalog_name = "automotive"
silver_schema = "silver"
gold_schema = "gold"

silver_fuel_stations_table = f"{catalog_name}.{silver_schema}.alt_fuel_stations"
gold_alt_fuel_infrastructure_table = f"{catalog_name}.{gold_schema}.alt_fuel_infrastructure"

## Read silver table

Load the cleaned alternative fuel stations dataset from the silver layer.

In [0]:
df_fuel_stations_silver = spark.table(silver_fuel_stations_table)

## Build analytical aggregation

Aggregate station data by state and fuel type to create a simplified infrastructure analysis table.

In [0]:
df_gold_alt_fuel_infrastructure = (
    df_fuel_stations_silver
    .groupBy("state", "fuel_type_code")
    .agg(
        F.count("*").alias("station_count"),
        F.sum(F.coalesce(F.col("ev_level1_evse_num"), F.lit(0))).alias("level_1_ev_ports"),
        F.sum(F.coalesce(F.col("ev_level2_evse_num"), F.lit(0))).alias("level_2_ev_ports"),
        F.sum(F.coalesce(F.col("ev_dc_fast_count"), F.lit(0))).alias("dc_fast_ev_ports")
    )
)

## Reorder columns

Move the most relevant analytical fields to the front.

In [0]:
preferred_column_order = [
    "state",
    "fuel_type_code",
    "station_count",
    "level_1_ev_ports",
    "level_2_ev_ports",
    "dc_fast_ev_ports"
]

final_columns = [c for c in preferred_column_order if c in df_gold_alt_fuel_infrastructure.columns] + [
    c for c in df_gold_alt_fuel_infrastructure.columns if c not in preferred_column_order
]

df_gold_alt_fuel_infrastructure = df_gold_alt_fuel_infrastructure.select(*final_columns)

## Persist gold table

Save the aggregated infrastructure analysis table into the gold layer as a Delta table.

In [0]:
(
    df_gold_alt_fuel_infrastructure.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(gold_alt_fuel_infrastructure_table)
)

## Final note

This gold table provides a simplified analytical view of alternative fuel infrastructure by state and fuel type, supporting regional analysis of infrastructure readiness.

## Data preview

Display statements are included for demonstration purposes to provide a quick visual inspection of the data in each layer.

These previews are intended to support the assessment by allowing easy validation of the transformations applied throughout the pipeline.

In [0]:
display(spark.table(gold_alt_fuel_infrastructure_table))

state,fuel_type_code,station_count,level_1_ev_ports,level_2_ev_ports,dc_fast_ev_ports
NJ,LPG,16,0,0,0
ND,LPG,19,0,0,0
ND,E85,41,0,0,0
HI,ELEC,452,28,1006,189
NM,ELEC,511,14,762,562
HI,HY,2,0,0,0
ID,ELEC,273,17,479,280
GA,CNG,47,0,0,0
BC,ELEC,2964,29,6578,2477
PE,ELEC,179,0,330,23
